> **Public copy.** This step reads the raw Jira export and the test trackers, which stay private, so it cannot be run from this repository. The outputs below are from the original run. After that run, product, vendor and system names and release numbers in the code, text and tables were replaced with the public codes used everywhere in this repository (Vendor A to E, releases such as 1.51, System tag 1 to 11 and so on), so every count is exactly as computed.


# 05 Components for every defect

**Goal:** give every defect its components, using the pattern Jagan set by hand.

**How the pattern was built (23 Nov 2026):**
1. Jagan labelled 50 defects by hand (a random sample spread over every component and year).
2. The rules in `scripts/hs_tags.py` were written from his answers.
3. The rules filled the next 100 defects, which Jagan checked. **Before seeing his corrections, the rules got the main component right for about 9 in 10 of the 97 he checked, and everything right (main component and all tags) for 82%.**
4. His corrections were added, and the rules were applied to all defects.
5. A final check on a fresh random 50 (never seen before): **43 of 50 fully right (86%), and the main component right for 45 of 50 (90%).** His last corrections were added too.
6. The defects the rules still could not place (156 "Unknown") went back to Jagan. He placed 135 of them. His answers added two main components (**EPP** and **Home delivery service**) and new detail tags (Customer display, System tag 8, Cache, System tag 6, User maintenance), and taught new rules (test tickets and duplicates are not product defects; Gryphon scanners and pole lights are hardware; Jenkins, SentinelOne, McAfee and login failures are environment; customer displays and system tag 7 are till screens). For these defects his main and other components replace the rules', and the rules' keyword detail tags are kept.
7. Every defect Jagan labelled by hand keeps his answer (`data/private/component_overrides.csv`): a human answer always beats the rules.

Rules agreed along the way: Vendor D covers Vendor D, left in trolley, active and silent mode; vendor names are never tags; a product name followed by a number is that product's version; perf defects are out of scope; EUC and environment are one group, with EUC as a tag.
The model uses the main components (about 15 groups, so each has enough defects to learn from); the dashboard shows all the detail tags, because QA stakeholders want the detail.

In [1]:
import sqlite3, sys
from pathlib import Path
import pandas as pd

# Works whether you run from the project folder or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
WORK = ROOT / 'data' / 'private' / 'hindsight_work.sqlite'    # Jira export (read only)
CLEAN = ROOT / 'data' / 'private' / 'hindsight_clean.sqlite'  # our cleaned tables
pd.set_option('display.width', 200)
print('Reading from:', WORK)
from hs_tags import tag_defect
from hs_overrides import apply, MAIN
with sqlite3.connect(CLEAN) as cc:
    d = pd.read_sql('select key, summary, labels, counts_for_risk from defects', cc)
ovr_path = ROOT / 'data' / 'private' / 'component_overrides.csv'
ovr = pd.read_csv(ovr_path).fillna('').set_index('key').to_dict('index') if ovr_path.exists() else {}
print(len(d), 'defects;', len(ovr), 'hand-labelled answers')

Reading from: data/private/hindsight_work.sqlite
3152 defects; 331 hand-labelled answers


In [2]:
rows, prim, src = [], {}, {}
for r in d.itertuples():
    p, others, labs = tag_defect(r.labels, r.summary)
    o = ovr.get(r.key)
    p, others, labs = apply(p, others, labs, o)
    prim[r.key] = p
    src[r.key] = o['source'] if o else 'rules'
    rows.append({'key': r.key, 'name': p, 'kind': 'main component', 'is_primary': 1})
    for x in others:
        if x != p:
            rows.append({'key': r.key, 'name': x, 'kind': 'component' if x in MAIN else 'detail tag', 'is_primary': 0})
    for l in labs:
        rows.append({'key': r.key, 'name': l, 'kind': 'label tag', 'is_primary': 0})
tags = pd.DataFrame(rows).drop_duplicates(['key', 'name'])
d['primary_component'] = d.key.map(prim)
d['component_source'] = d.key.map(src)
print(d.component_source.value_counts().to_string())
d.primary_component.value_counts().to_frame('defects')

component_source
rules                    2821
unknown list              135
batch 2 confirmed          79
hand label (first 50)      50
spot-check confirmed       43
batch 2 correction         17
spot-check correction       7


,defects
primary_component,
Scan and go (handsets),767
Self-checkout (SCO),489
Environment and infrastructure,326
Payments and PED,323
"Promotions, pricing and loyalty",316
Reporting,212
Till application (POS core),111
Out of scope (perf),102
Integrations and interfaces,94


In [3]:
comp = tags[tags.kind.isin(['main component', 'component'])]
per_defect = comp.groupby('key').size()
print('defects with 2 or more components:', int((per_defect > 1).sum()), f'({(per_defect > 1).mean():.0%})')
tags[tags.kind == 'detail tag'].name.value_counts().head(25).to_frame('defects')

defects with 2 or more components: 1361 (43%)


,defects
name,
UI,1051
Workflow / customer journey,312
Hardware,243
EUC,150
Store management app,123
XML,117
Coupons,116
Weighted items,92
Till flow,78


In [4]:
with sqlite3.connect(CLEAN) as cc:
    tags.to_sql('defect_components', cc, if_exists='replace', index=False)
    d[['key', 'primary_component', 'component_source']].to_sql('defect_primary_component', cc, if_exists='replace', index=False)
print('saved', len(tags), 'rows for', tags.key.nunique(), 'defects')

saved 10510 rows for 3152 defects
